# PCL + ACL Extraction and Cross-Check
This notebook pulls PCL (provision for credit losses) and ACL (allowance for credit losses) from two files,
then cross-checks PCL values to make sure the sources agree.

- PDF: quarterly financial report (PCL by business segment)
- Excel: supplemental report (PCL by business line + ACL by borrower type)


In [ ]:
# Imports
import re
from datetime import datetime
import pandas as pd
import pdfplumber


In [ ]:
# File paths (update if your folder differs)
pdf_path = '/Users/fredericzhang/Documents/Study/CapStone/Case_1/Q325_Shareholders_Report-EN.pdf'
excel_path = '/Users/fredericzhang/Documents/Study/CapStone/Case_1/2025Q3.xlsx'
output_path = '/Users/fredericzhang/Documents/Study/CapStone/Case_1/PCL_merged_output.xlsx'
crosscheck_path = '/Users/fredericzhang/Documents/Study/CapStone/Case_1/pcl_crosscheck_report.xlsx'


In [ ]:
# Business segments used in the PDF table
segments = [
    'Canadian Banking',
    'International Banking',
    'Wealth Management',
    'Banking and Markets',
    'Other',
    'Total',
]


In [ ]:
# Helper: parse a 'Provision for credit losses' line from the PDF
# We keep dashes so segment alignment is correct (e.g., Other may be a dash).
def parse_pcl_line(line):
    line_norm = line.replace('–', '-').replace('—', '-')
    tokens = re.findall(r'\(\d+[\d,]*\)|-?\d[\d,]*|-', line_norm)
    values = []
    for t in tokens:
        t = t.strip()
        if t == '-':
            values.append(None)
            continue
        if t.startswith('(') and t.endswith(')'):
            t = '-' + t[1:-1]
        values.append(t.replace(',', ''))
    # The first token is from the label if it looks numeric; skip it if needed
    # Keep only the last 6 values so alignment matches segments.
    if len(values) > 6:
        values = values[-6:]
    if len(values) < 6:
        values += [None] * (6 - len(values))
    return values


In [ ]:
# --- Extract PCL by segment from PDF ---
blocks = []
with pdfplumber.open(pdf_path) as pdf:
    # Pages 84 and 85 contain the segment PCL rows in this report
    for page_num in [84, 85]:
        page = pdf.pages[page_num - 1]
        text = page.extract_text() or ''
        lines = [ln.strip() for ln in text.split('\n') if ln.strip()]
        current = None
        for ln in lines:
            m = re.match(r'^For the (three|nine) months ended (.+)$', ln)
            if m:
                if current:
                    blocks.append(current)
                current = {
                    'period_type': m.group(1),
                    'period_end_raw': m.group(2),
                    'lines': []
                }
                continue
            if current:
                current['lines'].append(ln)
        if current:
            blocks.append(current)

pcl_rows_pdf = []
for block in blocks:
    # Parse date to ISO (YYYY-MM-DD); keep raw if parsing fails
    period_end_raw = block['period_end_raw']
    try:
        period_end = datetime.strptime(period_end_raw.replace(',', ''), '%B %d %Y').date().isoformat()
    except ValueError:
        period_end = period_end_raw

    for ln in block['lines']:
        if ln.startswith('Provision for credit losses'):
            values = parse_pcl_line(ln)
            for seg, val in zip(segments, values):
                pcl_rows_pdf.append({
                    'metric': 'Provision for credit losses',
                    'period_type': block['period_type'],
                    'period_end': period_end,
                    'segment': seg,
                    'value': val,
                    'source': 'Q325_Shareholders_Report-EN.pdf'
                })

pcl_pdf_df = pd.DataFrame(pcl_rows_pdf)
pcl_pdf_df.head()


In [ ]:
# --- Extract PCL by business line from Excel (sheet 23) ---
# This sheet includes multiple PCL sections; we only use the first section
# titled 'Total PCL ($ millions)' and stop before the next section.
pcl_excel_rows = []
df = pd.read_excel(excel_path, sheet_name='23', header=None)

# Map quarter labels in the sheet to period_end dates
quarter_to_date = {
    'Q3/25': '2025-07-31',
    'Q2/25': '2025-04-30',
    'Q1/25': '2025-01-31',
    'Q4/24': '2024-10-31',
    'Q3/24': '2024-07-31',
}

# Find header row containing quarter labels
header_row_idx = None
for i in range(0, 8):
    row_vals = df.iloc[i].tolist()
    if any(isinstance(v, str) and v.strip() in quarter_to_date for v in row_vals):
        header_row_idx = i
        break

# Find the 'Total PCL ($ millions)' section start
start_idx = None
for i in range(0, len(df)):
    cell = df.iloc[i, 0]
    if isinstance(cell, str) and 'Total PCL ($ millions)' in cell:
        start_idx = i + 1
        break

# Map Excel labels to PDF segment names
target_labels = {
    'Canadian Banking': 'Canadian Banking',
    'International Banking': 'International Banking',
    'Global Wealth Management': 'Wealth Management',
    'Global Banking and Markets': 'Banking and Markets',
    'Other': 'Other',
    'Total PCL': 'Total',
}

if header_row_idx is None or start_idx is None:
    raise ValueError('Could not locate PCL table in sheet 23')

# Build list of quarter column groups (every 3 columns: Stage 1&2, Stage 3, Total PCL)
quarter_cols = []
header_row = df.iloc[header_row_idx]
for col_idx, val in header_row.items():
    if isinstance(val, str) and val.strip() in quarter_to_date:
        quarter_cols.append((col_idx, val.strip()))

# Parse rows until next section starts
for i in range(start_idx, len(df)):
    label = df.iloc[i, 0]
    if not isinstance(label, str) or not label.strip():
        if i > start_idx:
            break
        continue
    label = label.strip()
    if label.startswith('PCL on loans') or 'Provision for Credit Losses as a %' in label:
        break
    if label not in target_labels:
        continue

    for col_start, quarter_label in quarter_cols:
        total_pcl = df.iloc[i, col_start + 2]
        if pd.isna(total_pcl):
            continue
        # Clean numbers stored as strings
        if isinstance(total_pcl, str):
            total_pcl = total_pcl.replace(',', '').strip()
            if total_pcl in ('-', '–'):
                total_pcl = None
        pcl_excel_rows.append({
            'metric': 'Provision for credit losses',
            'period_end': quarter_to_date.get(quarter_label, quarter_label),
            'segment': target_labels[label],
            'value': total_pcl,
            'source': '2025Q3.xlsx'
        })

pcl_excel_df = pd.DataFrame(pcl_excel_rows)
pcl_excel_df.head()


In [ ]:
# --- Extract ACL (Allowance for Credit Losses) from Excel (sheet 22) ---
# This is a borrower-type breakdown (not directly comparable to PDF PCL).
df_acl = pd.read_excel(excel_path, sheet_name='22', header=None)

# Find the section by title
title_idx = None
matches = df_acl[0].astype(str).str.contains('Impaired Loans by Type of Borrower', na=False)
if matches.any():
    title_idx = matches[matches].index[0]

acl_rows = []
if title_idx is not None:
    # Find the row that contains the dates
    date_row_idx = None
    date_pattern = re.compile(r'^[A-Za-z]+ \d{1,2}, \d{4}$')
    for i in range(title_idx + 1, title_idx + 5):
        row = df_acl.iloc[i].tolist()
        if any(isinstance(x, str) and date_pattern.match(x.strip()) for x in row):
            date_row_idx = i
            break

    # Subheader row with gross/stage3/net
    subheader_row_idx = None
    for i in range(date_row_idx + 1, date_row_idx + 4):
        cell = df_acl.iloc[i, 0]
        if isinstance(cell, str) and '($ millions' in cell:
            subheader_row_idx = i
            break
    if subheader_row_idx is None:
        subheader_row_idx = date_row_idx + 1

    # Determine the end of the table
    end_idx = None
    for i in range(subheader_row_idx + 1, len(df_acl)):
        cell = df_acl.iloc[i, 0]
        if isinstance(cell, str) and 'Impaired Loans, Net of Related Allowances' in cell:
            end_idx = i
            break
    if end_idx is None:
        end_idx = len(df_acl)

    data = df_acl.iloc[subheader_row_idx + 1:end_idx].copy()

    # Identify date columns (every 3 columns per date)
    date_cols = []
    date_row = df_acl.iloc[date_row_idx]
    for col_idx, val in date_row.items():
        if isinstance(val, str) and date_pattern.match(val.strip()):
            date_cols.append((col_idx, val.strip()))

    def clean_num(x):
        if pd.isna(x):
            return None
        if isinstance(x, str):
            x = x.strip()
            if x in ('-', '–'):
                return None
            x = x.replace(',', '')
        return x

    for _, row in data.iterrows():
        label = row.iloc[0]
        if not isinstance(label, str) or not label.strip():
            continue
        row_vals = row.iloc[1:]
        if row_vals.isna().all():
            continue
        label = label.strip()
        for col_start, date_str in date_cols:
            gross = row.iloc[col_start]
            stage3 = row.iloc[col_start + 1] if col_start + 1 < len(row) else None
            net = row.iloc[col_start + 2] if col_start + 2 < len(row) else None
            if pd.isna(gross) and pd.isna(stage3) and pd.isna(net):
                continue
            acl_rows.append({
                'metric': 'Allowance for Credit Losses',
                'borrower_type': label,
                'period_end': datetime.strptime(date_str, '%B %d, %Y').date().isoformat(),
                'gross': clean_num(gross),
                'stage_3': clean_num(stage3),
                'net': clean_num(net),
                'source': '2025Q3.xlsx'
            })

acl_excel_df = pd.DataFrame(acl_rows)
acl_excel_df.head()


In [ ]:
# --- Cross-check PCL between PDF and Excel ---
# Rules: absolute diff <= 1 (in $ millions) OR relative diff <= 1%
# We compare quarterly (three months) PCL only.
pcl_pdf_check = pcl_pdf_df[pcl_pdf_df['period_type'] == 'three'][['period_end', 'segment', 'metric', 'value']].copy()
pcl_excel_check = pcl_excel_df[['period_end', 'segment', 'metric', 'value']].copy()

# Convert values to numeric
pcl_pdf_check['value'] = pd.to_numeric(pcl_pdf_check['value'], errors='coerce')
pcl_excel_check['value'] = pd.to_numeric(pcl_excel_check['value'], errors='coerce')

merged = pcl_pdf_check.merge(
    pcl_excel_check,
    on=['period_end', 'segment', 'metric'],
    how='inner',
    suffixes=('_pdf', '_excel')
)

# If one source is missing and the other is very close to zero,
# treat it as a pass but record a note.
merged['note'] = ''
missing_pdf = merged['value_pdf'].isna() & merged['value_excel'].notna()
missing_excel = merged['value_excel'].isna() & merged['value_pdf'].notna()

near_zero_pdf = missing_excel & (merged['value_pdf'].abs() <= 1)
near_zero_excel = missing_pdf & (merged['value_excel'].abs() <= 1)

merged.loc[near_zero_pdf, 'note'] = 'PDF missing; treated as 0 within tolerance'
merged.loc[near_zero_excel, 'note'] = 'Excel missing; treated as 0 within tolerance'

# Standard diff check (ignore rows where both are NaN)
merged['diff'] = (merged['value_pdf'] - merged['value_excel']).abs()
denom = merged[['value_pdf', 'value_excel']].abs().max(axis=1).replace(0, 1)
merged['rel_diff'] = merged['diff'] / denom

merged['pass'] = (merged['diff'] <= 1) | (merged['rel_diff'] <= 0.01)
merged.loc[near_zero_pdf | near_zero_excel, 'pass'] = True

merged.to_excel(crosscheck_path, index=False)
print('Wrote cross-check report:', crosscheck_path)
print('Rows:', len(merged), 'Failed:', (~merged['pass']).sum())


In [ ]:
# --- Write combined output ---
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    pcl_pdf_df.to_excel(writer, sheet_name='PCL_PDF', index=False)
    pcl_excel_df.to_excel(writer, sheet_name='PCL_Excel', index=False)
    acl_excel_df.to_excel(writer, sheet_name='ACL_Excel', index=False)

print('Wrote:', output_path)
